In [9]:
from pathlib import Path
import shutil
import random
import os
from collections import Counter

In [10]:
SOURCE = Path(
    r"D:\Projects\DeepLearning\YOLO_Disaster_Project\datasets\yolo_dataset"
)

BALANCED = Path(
    r"D:\Projects\DeepLearning\YOLO_Disaster_Project\datasets\balanced_yolo_dataset"
)

In [11]:
for split in ["train", "val"]:

    (BALANCED / "images" / split).mkdir(parents=True, exist_ok=True)

    (BALANCED / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Balanced dataset folders created")

Balanced dataset folders created


In [12]:
removed = 0

for split in ["train", "val"]:

    label_dir = SOURCE / "labels" / split
    image_dir = SOURCE / "images" / split

    for txt_file in label_dir.glob("*.txt"):

        if os.path.getsize(txt_file) == 0:

            image_file = image_dir / f"{txt_file.stem}.png"

            txt_file.unlink()

            if image_file.exists():
                image_file.unlink()

            removed += 1

print("Removed empty files:", removed)

Removed empty files: 558


In [13]:
MAX_NO_DAMAGE = 10000

random.seed(42)

CLASS_NAMES = {
    0: "no-damage",
    1: "minor-damage",
    2: "major-damage",
    3: "destroyed"
}

for split in ["train", "val"]:

    image_dir = SOURCE / "images" / split
    label_dir = SOURCE / "labels" / split

    no_damage = []
    damaged = []

    for txt_file in label_dir.glob("*.txt"):

        with open(txt_file, "r") as f:

            lines = [line.strip() for line in f.readlines() if line.strip()]

        if len(lines) == 0:
            continue

        classes = [int(line.split()[0]) for line in lines]

        if all(c == 0 for c in classes):

            no_damage.append(txt_file)

        else:

            damaged.append(txt_file)

    print(f"\n{split.upper()}")

    print("Original no-damage:", len(no_damage))
    print("Damaged:", len(damaged))

    random.shuffle(no_damage)

    no_damage = no_damage[:MAX_NO_DAMAGE]

    final_files = no_damage + damaged

    print("Final dataset size:", len(final_files))

    for txt_file in final_files:

        image_file = image_dir / f"{txt_file.stem}.png"

        shutil.copy(
            image_file,
            BALANCED / "images" / split / image_file.name
        )

        shutil.copy(
            txt_file,
            BALANCED / "labels" / split / txt_file.name
        )

print("\nBalanced dataset created successfully")


TRAIN
Original no-damage: 594
Damaged: 1203
Final dataset size: 1797

VAL
Original no-damage: 146
Damaged: 298
Final dataset size: 444

Balanced dataset created successfully


In [14]:
yaml_text = """

path: D:/Projects/DeepLearning/YOLO_Disaster_Project/datasets/balanced_yolo_dataset

train: images/train
val: images/val

names:
  0: no-damage
  1: minor-damage
  2: major-damage
  3: destroyed

"""

with open("balanced_data.yaml", "w") as f:

    f.write(yaml_text)

print("balanced_data.yaml created")

balanced_data.yaml created


In [15]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="balanced_data.yaml",
    epochs=60,
    imgsz=512,
    batch=1,
    workers=0,
    device=0,
    patience=20,
    cache=False
)

New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.21  Python-3.10.11 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=balanced_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=t

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002386F269EA0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([0.        , 0.001001  , 0.002002  , 0.003003  , 0.004004  ,
       0.00500501, 0.00600601, 0.00700701, 0.00800801, 0.00900901,
       0.01001001, 0.01101101, 0.01201201, 0.01301301, 0.01401401,
       0.01501502, 0.01601602, 0.01701702, 0.01801802, 0.01901902,
       0.02002002, 0.02102102, 0.02202202, 0.02302302, 0.02402402,
       0.02502503, 0.02602603, 0.02702703, 0.02802803, 0.02902903,
       0.03003003, 0.03103103, 0.03203203, 0.03303303, 0.03403403,
       0.03503504, 0.03603604, 0.03703704, 0.03803804, 0.03903904,
       0.04004004, 0.04104104, 0.04204204, 0.04304304, 0.04404404,
       0.04504505, 0.04604605, 0.04704705,

In [7]:
model = YOLO("yolov8s.pt")

model.train(
    data="data.yaml",
    epochs=30,
    imgsz=416,
    batch=2,
    workers=0,
    device=0
)

New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.21  Python-3.10.11 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train5, nb

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002375225FBE0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([0.        , 0.001001  , 0.002002  , 0.003003  , 0.004004  ,
       0.00500501, 0.00600601, 0.00700701, 0.00800801, 0.00900901,
       0.01001001, 0.01101101, 0.01201201, 0.01301301, 0.01401401,
       0.01501502, 0.01601602, 0.01701702, 0.01801802, 0.01901902,
       0.02002002, 0.02102102, 0.02202202, 0.02302302, 0.02402402,
       0.02502503, 0.02602603, 0.02702703, 0.02802803, 0.02902903,
       0.03003003, 0.03103103, 0.03203203, 0.03303303, 0.03403403,
       0.03503504, 0.03603604, 0.03703704, 0.03803804, 0.03903904,
       0.04004004, 0.04104104, 0.04204204, 0.04304304, 0.04404404,
       0.04504505, 0.04604605, 0.04704705,

In [8]:
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt

LABEL_DIR = Path(
    r"D:\Projects\DeepLearning\YOLO_Disaster_Project\datasets\yolo_dataset\labels\train"
)

class_counts = Counter()

empty_files = 0
total_boxes = 0
multi_class_images = 0

for txt_file in LABEL_DIR.glob("*.txt"):

    with open(txt_file, "r") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    if len(lines) == 0:
        empty_files += 1
        continue

    image_classes = set()

    for line in lines:

        class_id = int(line.split()[0])

        class_counts[class_id] += 1
        total_boxes += 1

        image_classes.add(class_id)

    if len(image_classes) > 1:
        multi_class_images += 1

CLASS_NAMES = {
    0: "no-damage",
    1: "minor-damage",
    2: "major-damage",
    3: "destroyed"
}

print("\n===== DATASET ANALYSIS =====")

print("\nClass counts:")
for class_id, count in class_counts.items():
    print(f"{CLASS_NAMES[class_id]}: {count}")

print("\nTotal bounding boxes:", total_boxes)

print("Empty label files:", empty_files)

print("Images with multiple classes:", multi_class_images)

# BAR CHART

names = [CLASS_NAMES[k] for k in class_counts.keys()]
values = [class_counts[k] for k in class_counts.keys()]

plt.figure(figsize=(8,5))
plt.bar(names, values)

plt.title("Class Distribution")
plt.ylabel("Number of Objects")

plt.show()


===== DATASET ANALYSIS =====

Class counts:
no-damage: 92162
minor-damage: 11866
destroyed: 11112
major-damage: 11123

Total bounding boxes: 126263
Empty label files: 442
Images with multiple classes: 991


<Figure size 800x500 with 1 Axes>